# Video Face Swap - 1 người + **thay tóc** (inswapper + InsightFace + MediaPipe)

**Logic:** 1 video mẫu (một người chuyển động) + 1 ảnh khuôn mặt → video mới giữ nguyên
chuyển động/nền gốc, thay **khuôn mặt** *và* **mái tóc** bằng của người trong ảnh.

Notebook này = `video_face_swap_1nguoi.ipynb` + **một tầng mới: chuyển tóc** (mục 5b).
Toàn bộ phần vá môi trường đã debug kỹ ở bản gốc được giữ nguyên.

## Vì sao phải làm riêng phần tóc?

`inswapper_128` **cố tình không đụng tới tóc**. Nó chỉ warp vùng mặt đã align 128×128 (từ chân
mày tới cằm) rồi dán ngược về. Đó là lý do kết quả swap luôn giữ đúng kiểu tóc của người trong
video — muốn đổi tóc thì phải tự làm, không có công tắc nào bật lên được.

## Đường đi của một khung hình

```
frame gốc
  → InsightFace detect  (bbox + 5 điểm mốc)
  → inswapper swap mặt  (paste_back)
  → GFPGAN làm nét mặt  (tuỳ chọn)
  → THAY TÓC:  warp tóc-nguồn theo 5 điểm mốc → xoá tóc cũ thừa → blend  ← phần mới
  → ffmpeg
```

**Thay tóc chạy CUỐI CÙNG** chứ không phải trước khi swap. Lý do: GFPGAN "sáng tác" lại chi tiết
trên ô crop quanh mặt (ô này lấn sang cả tóc vì `GFPGAN_PAD = 0.4`). Dán tóc trước thì GFPGAN sẽ
vẽ lại đúng đường ghép tóc vừa dán, mỗi frame một kiểu → nhấp nháy ngay ở chỗ dễ thấy nhất.

## Ba bước của phần tóc

| Bước | Làm gì | Bằng gì |
|---|---|---|
| **Segment** | tách vùng tóc khỏi ảnh nguồn (1 lần) và khỏi mỗi frame video | MediaPipe Image Segmenter, model `selfie_multiclass_256x256` — có sẵn lớp `hair`, tải thẳng từ Google, không cần checkpoint lạ |
| **Warp** | đưa tóc-nguồn về đúng vị trí/kích thước/góc nghiêng của cái đầu trong frame | phép **đồng dạng** khớp 5 điểm mốc (Umeyama), làm mượt theo thời gian bằng EMA |
| **Blend** | dán tóc mới, giấu tóc cũ, khớp ánh sáng | alpha mềm + mask bảo vệ mặt + inpaint phần tóc cũ thừa + hoà sáng kênh L |

## Những chỗ đã xử lý sẵn (đọc trước khi sửa tham số)

- **Không đè lên mặt.** Mask "face-skin" của MediaPipe nhân với một dốc mềm vuông góc trục hai
  mắt: tóc mái được phép phủ trán tới mức `HAIR_COVER_FOREHEAD`, nhưng **không bao giờ** trườn
  xuống mắt/mũi/miệng — kể cả khi ảnh nguồn tóc dài trùm mặt.
- **Tóc cũ thò ra.** Tóc nguồn ngắn hơn tóc trong video thì phần thừa vẫn lộ. `HAIR_REMOVE_LEFTOVER`
  inpaint vùng đó theo nền trước khi dán tóc mới.
- **Rung theo thời gian.** Landmark nhảy vài pixel mỗi frame; tóc là mảng lớn nên rung rất rõ.
  Đã làm mượt **tham số biến đổi** (không phải làm mượt ảnh) + EMA mask phân vùng.
- **Quay đầu.** Tóc-nguồn là ảnh 2D chính diện, dán lên đầu đang quay nghiêng sẽ vỡ. Có đo độ
  chính diện từ 5 điểm mốc và **mờ dần tóc** khi quay quá nhiều, thay vì dán bừa.
- **Không tất định.** Dùng công thức đóng Umeyama chứ không phải `cv2.estimateAffinePartial2D`
  (RANSAC/LMEDS với đúng 5 điểm cho kết quả khác nhau giữa các frame → tóc giật).

⚠️ Lưu ý: face-swap + hair-swap có thể bị dùng sai mục đích (deepfake giả mạo người khác mà
không có sự đồng ý). Chỉ dùng với ảnh/video của chính bạn hoặc người đã đồng ý, và cân nhắc
gắn watermark/disclosure khi xuất bản sản phẩm thật.

**Trước khi chạy:** Runtime > Change runtime type > chọn GPU (T4).

## 0. Cấu hình

**Hai công tắc lớn** — mỗi cái tự tắt luôn cả phần cài đặt lẫn tải model của nó, không chỉ tắt
lúc chạy:

| | `True` | `False` |
|---|---|---|
| `USE_GFPGAN` | cài `gfpgan/facexlib/basicsr` (phải vá source), tải `GFPGANv1.4.pth` (~340MB), làm nét crop quanh mặt mỗi frame | không cài, không tải, mặt giữ chất lượng thô của inswapper |
| `USE_HAIR` | cài `mediapipe`, tải `selfie_multiclass_256x256.tflite` (~16MB), thay tóc mỗi frame | không cài, không tải, giữ nguyên tóc trong video |

Bên dưới còn một nhóm tham số tinh chỉnh riêng cho tóc. **Không cần hiểu hết ngay**: chạy tới
mục 6.1 (thử 1 frame) sẽ thấy ngay kết quả, chỉnh rồi chạy lại mỗi cell đó — vài giây một vòng,
không phải xử lý cả video.

### Ba tham số đáng chỉnh nhất

- **`HAIR_COVER_FOREHEAD`** (0→1): tóc mái được phủ xuống trán bao xa. `0` = không chạm mặt,
  `1` = xuống sát chân mày. Ảnh nguồn có mái thì tăng, vuốt ngược để trán trần thì giảm.
- **`HAIR_REMOVE_LEFTOVER`**: bật khi tóc nguồn **ngắn hơn** tóc trong video (không bật thì tóc
  cũ thò ra hai bên). Tóc nguồn dài hơn thì tắt đi cho nhanh và sạch nền.
- **`HAIR_MAX_TURN`**: ngưỡng quay đầu bắt đầu mờ tóc. Video quay nghiêng nhiều mà tóc biến mất
  liên tục thì nới rộng, nhưng đổi lại tóc sẽ méo ở những frame đó.

### Về tốc độ

Phần tóc thêm **một lần segment MediaPipe trên vùng đầu** (~256×256, CPU) + vài phép warp/blend
mỗi frame. Rẻ hơn GFPGAN nhiều. `HAIR_REMOVE_LEFTOVER = True` là khoản đắt nhất trong nhóm này
(`cv2.inpaint`, chỉ chạy khi thật sự có tóc cũ thừa). Cell cuối mục 6 in thời gian trung bình
**từng bước** (detect / swap / làm nét / tóc) để bạn biết chính xác cái nào tốn, thay vì đoán.

In [ ]:
# ===================== CÔNG TẮC CHÍNH =====================
USE_GFPGAN = True    # True  = swap xong làm nét mặt (đẹp hơn, chậm hơn)
                     # False = chỉ swap, không làm nét (nhanh hơn, ít lỗi cài đặt hơn)

USE_HAIR   = True    # True  = thay luôn mái tóc bằng tóc trong ảnh nguồn
                     # False = giữ nguyên tóc của người trong video (như notebook gốc)
# ==========================================================


# ============ TINH CHỈNH PHẦN TÓC (chỉ có tác dụng khi USE_HAIR = True) ============
# Chỉnh xong thì chạy lại cell 6.1 (thử 1 frame) để xem ngay, không cần chạy cả video.

HAIR_STRENGTH        = 1.0    # độ đậm tổng thể của tóc mới. <1 = pha loãng với tóc cũ
                              # (0.7-0.8 hữu ích khi tóc nguồn và tóc cũ khác nhau quá đột ngột)

HAIR_COVER_FOREHEAD  = 0.35   # tóc mái phủ trán tới đâu: 0 = không chạm mặt chút nào,
                              # 1 = xuống sát chân mày. Không bao giờ vượt quá đường mắt.

HAIR_REMOVE_LEFTOVER = True   # inpaint phần tóc CŨ thò ra ngoài vùng tóc mới.
                              # Bật khi tóc nguồn ngắn hơn tóc trong video.

HAIR_LIGHT_MATCH     = 0.7    # 0-1: kéo độ sáng tóc mới về theo ánh sáng của frame
                              # (đo bằng chênh lệch độ sáng da mặt giữa ảnh nguồn và frame).
                              # 0 = giữ nguyên màu ảnh gốc, 1 = bám hoàn toàn theo frame.

HAIR_MAX_TURN        = (0.16, 0.38)   # (bắt đầu mờ, mất hẳn) theo độ lệch mũi so với tâm hai
                              # mắt, chuẩn hoá theo khoảng cách hai mắt. ~0.16 ≈ quay 20-25°,
                              # ~0.38 ≈ gần nghiêng hẳn. Tóc 2D không dán đúng lên đầu quay.

HAIR_ROI             = (2.6, 3.2, 4.5)   # vùng làm việc quanh đầu: (ngang, trên, dưới),
                              # tính theo khoảng cách hai mắt. Mọi thứ NGOÀI hộp này không được
                              # chạm tới -> tóc CŨ dài quá hộp sẽ không bị xoá. Người trong video
                              # tóc dài ngang lưng thì tăng số thứ 3 lên (6.0-7.0).

HAIR_SMOOTH          = 0.55   # làm mượt phép biến đổi theo thời gian (EMA).
HAIR_MASK_SMOOTH     = 0.5    # làm mượt mask phân vùng theo thời gian.
                              # Cả hai: 0 = không mượt (rung), 0.8 = rất mượt (trễ, bóng ma khi
                              # chuyển động nhanh). Chỉ đụng vào khi thấy rung/trễ rõ.

HAIR_WARP_MODE       = 'similarity'   # 'similarity' = xoay + phóng đều + tịnh tiến (ổn định).
                              # 'affine' = thêm co ngang/xô nghiêng, bám đầu quay tốt hơn
                              # nhưng nhạy nhiễu landmark hơn -> dễ rung.

HAIR_THRESH          = 0.5    # ngưỡng nhận là "tóc" trên mask xác suất của MediaPipe.
                              # Giảm (0.35) nếu tóc nguồn bị hụt viền; tăng nếu ăn lẹm sang nền.

HAIR_FEATHER         = 0.025  # độ mềm mép tóc, tính theo tỉ lệ bề ngang đầu.
# ==================================================================================

print('USE_GFPGAN =', USE_GFPGAN)
if USE_GFPGAN:
    print('  -> cài gfpgan/facexlib/basicsr, tải thêm ~340MB weights, làm nét mặt mỗi frame.')
else:
    print('  -> bỏ qua toàn bộ phần làm nét: không cài, không tải, không chạy.')

print('USE_HAIR   =', USE_HAIR)
if USE_HAIR:
    print('  -> cài mediapipe, tải ~16MB model segment, thay tóc mỗi frame.')
    print(f'     phủ trán {HAIR_COVER_FOREHEAD:.2f} | xoá tóc cũ thừa: {HAIR_REMOVE_LEFTOVER}'
          f' | hoà sáng {HAIR_LIGHT_MATCH:.2f} | warp {HAIR_WARP_MODE!r}')
    print(f'     vùng quanh đầu (ngang/trên/dưới, theo khoảng cách hai mắt): {HAIR_ROI}')
else:
    print('  -> giữ nguyên tóc của người trong video.')

## 1. Cài đặt thư viện

In [ ]:
# Log phiên bản Python Colab đang cấp (để biết đang chạy bản nào)
import sys, platform
print('Python version:', sys.version)
print('Platform:', platform.platform())

In [ ]:
# Ghim setuptools < 82: bản setuptools mới (>=82) đã bỏ hẳn module 'distutils',
# trong khi basicsr (dependency của GFPGAN) và torch trên Colab vẫn cần distutils/setuptools cũ.
!pip install -q "setuptools==79.0.1" wheel
!pip install -q cython numpy

# --no-build-isolation: để insightface dùng đúng cython/numpy vừa cài ở trên,
# thay vì pip tự tạo môi trường tạm cô lập (không thấy cython) rồi build lỗi 'egg_info'.
!pip install -q --no-build-isolation insightface==0.7.3

# KHÔNG cài opencv-python-headless: Colab đã có sẵn opencv-python, mà hai package này
# dùng chung namespace `cv2` -> cài đè lên nhau hay để lại .so lẫn lộn gây lỗi khó hiểu.
# (insightface cũng khai báo opencv-python là dependency nên chắc chắn có cv2.)

# gfpgan/facexlib chỉ cần khi làm nét. Tắt thì bỏ hẳn -> nhanh hơn và bớt một nguồn lỗi.
EXTRA_PKGS = 'gfpgan facexlib' if USE_GFPGAN else ''
print('Cài thêm:', EXTRA_PKGS or '(không có, USE_GFPGAN = False)')
!pip install -q onnxruntime-gpu {EXTRA_PKGS}

# libportaudio2: mediapipe import `sounddevice`, thư viện này ném OSError ngay lúc import nếu
# thiếu PortAudio trên máy -> `import mediapipe` chết dù package cài thành công. Ảnh Colab có
# lúc có lúc không, cài luôn cho chắc (vài trăm KB).
!apt-get -qq install -y ffmpeg libportaudio2 > /dev/null
print('Xong.')

### Cài `mediapipe` mà không đụng vào `cv2` (chỉ chạy khi `USE_HAIR = True`)

`pip install mediapipe` kéo theo **`opencv-contrib-python`**. Package đó chiếm đúng namespace
`cv2` mà `opencv-python` (Colab cài sẵn, insightface đang dùng) đang chiếm — cài đè lên nhau là
đúng cái lỗi `cv2` lẫn lộn mà notebook gốc đã ghi chú tránh ở cell trên. Nó cũng hay ghim lại
`numpy`/`protobuf`, làm hỏng ngược `onnxruntime` vừa cài.

Nên cell dưới cài `--no-deps` rồi **tự cài đúng những dependency thật sự cần** cho Image
Segmenter (`absl-py`, `attrs`, `flatbuffers`, `protobuf`, `jax` không cần). `cv2` thì dùng lại
bản Colab đã có.

Nếu bản `--no-deps` không import được, cell **tự fallback sang `pip install mediapipe` đầy đủ**
và in cảnh báo — vẫn chạy được, chỉ là môi trường bẩn hơn. Nếu cả hai đều hỏng thì
`HAIR_AVAILABLE = False` và pipeline chạy tiếp **không thay tóc** (giống cách `gfpgan` fallback),
chứ không làm chết cả notebook.

In [ ]:
import subprocess, sys, importlib

HAIR_AVAILABLE = False


def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', *args], capture_output=True, text=True)
    print(r.stdout[-2500:])
    print(r.stderr[-2500:])
    return r.returncode


def try_import_mediapipe():
    """Import thử cả `mediapipe` lẫn đúng submodule Image Segmenter sẽ dùng.

    Import mỗi `mediapipe` là chưa đủ: `mediapipe.tasks.python.vision` mới là chỗ ném lỗi khi
    thiếu dependency, mà nó chỉ được nạp khi gọi tới -> phải thử ngay tại đây.
    """
    importlib.invalidate_caches()
    try:
        import mediapipe as mp
        from mediapipe.tasks.python import vision as _vision   # noqa: F401
        print('mediapipe OK, version:', getattr(mp, '__version__', 'unknown'))
        return True
    except Exception as e:
        # Bắt Exception chứ không chỉ ImportError: thiếu PortAudio ném OSError,
        # lệch protobuf ném TypeError/AttributeError -> chỉ bắt ImportError thì cell crash.
        print(f'Chưa import được mediapipe: {type(e).__name__}: {e}')
        return False


if not USE_HAIR:
    print('USE_HAIR = False -> bỏ qua mediapipe (chỉ cần cho phần tóc).')
else:
    HAIR_AVAILABLE = try_import_mediapipe()

    if not HAIR_AVAILABLE:
        print('Cài mediapipe --no-deps (giữ nguyên cv2/numpy của Colab)...')
        pip('install', '-q', '--no-deps', 'mediapipe')
        # Dependency tối thiểu cho Tasks API. Không đụng numpy/opencv/protobuf-version.
        pip('install', '-q', 'absl-py', 'attrs', 'flatbuffers', 'sentencepiece', 'sounddevice')
        HAIR_AVAILABLE = try_import_mediapipe()

    if not HAIR_AVAILABLE:
        print()
        print('Bản --no-deps không chạy -> fallback: cài mediapipe đầy đủ.')
        print('CẢNH BÁO: bước này có thể cài đè opencv-contrib-python lên cv2 hiện có.')
        pip('install', 'mediapipe')
        HAIR_AVAILABLE = try_import_mediapipe()

        # Cài đè cv2 xong phải kiểm tra lại chính cv2 + onnxruntime, vì đó là thứ dễ vỡ nhất.
        for mod in ('cv2', 'onnxruntime'):
            try:
                m = importlib.import_module(mod)
                print(f'  {mod} vẫn OK, version:', getattr(m, '__version__', '?'))
            except Exception as e:
                print(f'  !! {mod} HỎNG sau khi cài mediapipe: {type(e).__name__}: {e}')
                print('     Runtime > Restart session rồi chạy lại từ đầu notebook.')

    if not HAIR_AVAILABLE:
        print()
        print('mediapipe không cài được -> sẽ chạy tiếp mà KHÔNG thay tóc.')
        print('Pipeline chính (swap mặt) vẫn chạy bình thường.')

### Fix riêng cho `basicsr` (chỉ chạy khi `USE_GFPGAN = True`)

`basicsr` có bug trong `setup.py`: dùng `exec(...)` rồi đọc `locals()['__version__']` để lấy
version. Ở Python 3.13, `exec()` trong 1 hàm không còn ghi ngược lại `locals()` đáng tin cậy
(thay đổi theo PEP 667) → lỗi `KeyError: '__version__'`. Cell dưới tải source về, patch đúng
chỗ này (`locals()` → `globals()`), rồi cài từ bản đã sửa.

`basicsr` chỉ là dependency của GFPGAN, nên `USE_GFPGAN = False` thì cell này không làm gì cả.

In [ ]:
import subprocess, sys, os, tarfile, urllib.request, json


def install_basicsr_patched():
    """Tải basicsr từ PyPI, vá bug PEP 667 trong setup.py, rồi cài từ source đã sửa."""
    os.makedirs('/tmp/basicsr_src', exist_ok=True)

    # Tải trực tiếp từ PyPI bằng urllib (KHÔNG dùng `pip download`, vì pip cũng phải chạy
    # setup.py egg_info để lấy metadata -> dính đúng bug KeyError trước khi kịp patch).
    with urllib.request.urlopen('https://pypi.org/pypi/basicsr/1.4.2/json') as resp:
        pkg_info = json.load(resp)

    sdist_url = None
    for url_info in pkg_info['urls']:
        if url_info['packagetype'] == 'sdist':
            sdist_url = url_info['url']
            break
    assert sdist_url is not None, 'Không tìm thấy sdist của basicsr trên PyPI.'

    tar_name = sdist_url.split('/')[-1]
    tar_path = f'/tmp/basicsr_src/{tar_name}'
    urllib.request.urlretrieve(sdist_url, tar_path)
    print(f'Đã tải: {tar_name}')

    extract_dir = '/tmp/basicsr_build'
    os.makedirs(extract_dir, exist_ok=True)
    with tarfile.open(tar_path) as tar:
        # Lấy tên thư mục gốc từ chính nội dung tar thay vì suy ra bằng tar_name.replace(...),
        # vì sdist trên PyPI không bắt buộc phải là .tar.gz.
        root_names = {m.name.split('/')[0] for m in tar.getmembers() if m.name.strip('./')}
        assert len(root_names) == 1, f'sdist có cấu trúc thư mục lạ: {root_names}'
        root_name = root_names.pop()
        # filter='data': Python 3.12+ deprecate extractall không có filter, 3.14 đổi default.
        tar.extractall(extract_dir, filter='data')

    pkg_dir = os.path.join(extract_dir, root_name)
    setup_py_path = os.path.join(pkg_dir, 'setup.py')
    assert os.path.exists(setup_py_path), f'Không thấy setup.py trong {pkg_dir}'

    with open(setup_py_path, 'r') as f:
        content = f.read()

    # Fix bug Python 3.13: exec() trong hàm không ghi ngược locals() đáng tin cậy
    content = content.replace(
        "exec(compile(f.read(), version_file, 'exec'))",
        "exec(compile(f.read(), version_file, 'exec'), globals())"
    )
    content = content.replace(
        "return locals()['__version__']",
        "return globals()['__version__']"
    )

    with open(setup_py_path, 'w') as f:
        f.write(content)

    print('Đã patch setup.py xong, tiến hành cài basicsr từ source đã sửa...')
    # sys.executable -m pip: đảm bảo cài vào đúng interpreter đang chạy notebook,
    # thay vì lệnh `pip` bất kỳ đứng đầu PATH.
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--no-build-isolation', pkg_dir],
        capture_output=True, text=True
    )
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])
    if r.returncode != 0:
        raise RuntimeError('Cài basicsr thất bại, xem log lỗi ở trên để biết nguyên nhân cụ thể.')
    print('Cài basicsr thành công.')


if USE_GFPGAN:
    install_basicsr_patched()
else:
    print('USE_GFPGAN = False -> bỏ qua basicsr (chỉ là dependency của GFPGAN).')

## 2. Tải model

Model `inswapper_128.onnx` không được host chính thức trên GitHub release nữa do vấn đề chính
sách, nên bạn cần tự tải và upload lên Google Drive của mình, hoặc dùng link mirror cộng đồng
(huggingface). Cell dưới thử tải từ 1 mirror phổ biến trên Hugging Face — nếu lỗi, bạn tải thủ
công rồi upload vào `/content/models/`.

| Model | Khi nào tải | Dung lượng |
|---|---|---|
| `inswapper_128.onnx` | luôn luôn | ~530 MB |
| `GFPGANv1.4.pth` | `USE_GFPGAN = True` | ~340 MB |
| `selfie_multiclass_256x256.tflite` | `USE_HAIR = True` | ~16 MB |

Model segment tóc tải **thẳng từ `storage.googleapis.com/mediapipe-models`** — kho chính thức
của Google cho MediaPipe Tasks, không phải mirror cộng đồng. Nó phân 6 lớp:
`0 background, 1 hair, 2 body-skin, 3 face-skin, 4 clothes, 5 others`. Notebook này dùng
**lớp 1 (hair)** để lấy tóc và **lớp 3 (face-skin)** để bảo vệ khuôn mặt + đo ánh sáng.

In [ ]:
import os, subprocess
os.makedirs('/content/models', exist_ok=True)

INSWAPPER_PATH = '/content/models/inswapper_128.onnx'
GFPGAN_PATH    = '/content/models/GFPGANv1.4.pth'
SEGMENTER_PATH = '/content/models/selfie_multiclass_256x256.tflite'

INSWAPPER_URL = 'https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx'
GFPGAN_URL    = 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth'
# Kho model chính thức của MediaPipe Tasks (Google), không phải mirror cộng đồng.
SEGMENTER_URL = ('https://storage.googleapis.com/mediapipe-models/image_segmenter/'
                 'selfie_multiclass_256x256/float32/latest/selfie_multiclass_256x256.tflite')


# Dùng subprocess thay vì `!wget`: lệnh `!` nằm trong khối `if` phụ thuộc vào chi tiết
# transform của IPython, còn subprocess thì chạy giống nhau ở mọi môi trường và
# trả về returncode để kiểm tra.
def download(url, path, min_mb, hint):
    """Tải file rồi KIỂM TRA DUNG LƯỢNG.

    wget vẫn coi là "thành công" khi server trả 404 / trang HTML -> phải tự kiểm tra.
    Nếu không, mãi tới cell load model mới nổ với lỗi onnx/torch/tflite rất khó đoán nguyên nhân.
    """
    print(f'Đang tải {os.path.basename(path)} ...')
    subprocess.run(['wget', '-q', '-O', path, url])
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    assert size_mb >= min_mb, (
        f'Tải {os.path.basename(path)} thất bại (chỉ {size_mb:.1f} MB, cần >= {min_mb} MB). {hint}'
    )
    print(f'OK  {os.path.basename(path)}: {size_mb:.1f} MB')


download(INSWAPPER_URL, INSWAPPER_PATH, 200,
         'Link mirror có thể đã chết -> tự tải rồi upload vào /content/models/.')

if USE_GFPGAN:
    download(GFPGAN_URL, GFPGAN_PATH, 300,
             'Kiểm tra lại link GitHub release của GFPGAN.')
else:
    print('Bỏ qua GFPGANv1.4.pth (~340MB) vì USE_GFPGAN = False.')

if USE_HAIR:
    download(SEGMENTER_URL, SEGMENTER_PATH, 5,
             'Kiểm tra lại đường dẫn trong kho mediapipe-models của Google.')
else:
    print('Bỏ qua selfie_multiclass_256x256.tflite vì USE_HAIR = False.')

!ls -lh /content/models/

## 3. Upload ảnh khuôn mặt + video mẫu

Chỉ còn **1 ảnh + 1 video**. Không cần quy ước ai là A ai là B, không cần lo upload nhầm thứ tự.

In [ ]:
from google.colab import files

def pick_one(uploaded, what):
    names = list(uploaded.keys())
    assert len(names) > 0, f'Chưa upload {what} (bấm Cancel?). Chạy lại cell này.'
    if len(names) > 1:
        print(f'  (đã upload {len(names)} file, dùng file đầu tiên: {names[0]})')
    return names[0]

print('>> Upload ảnh khuôn mặt (rõ mặt, chính diện càng tốt):')
source_face_path = pick_one(files.upload(), 'ảnh mặt')

print()
print('>> Upload video mẫu:')
source_video_path = pick_one(files.upload(), 'video mẫu')

print()
print(f'Ảnh mặt : {source_face_path}')
print(f'Video   : {source_video_path}')

## 4. Khởi tạo model face analysis + face swapper

### Fix riêng cho `onnxruntime-gpu` (có thể chưa có wheel cho Python bản mới trên Colab)

Cell dưới kiểm tra xem `onnxruntime` đã import được chưa. Nếu chưa, sẽ thử cài lại
`onnxruntime-gpu` với log đầy đủ; nếu vẫn thất bại (do chưa có wheel tương thích Python 3.13),
sẽ **fallback sang `onnxruntime` bản CPU** để pipeline vẫn chạy được (chỉ chậm hơn).

In [ ]:
import subprocess, sys, importlib

def try_import_onnxruntime():
    # Bắt Exception chứ không chỉ ImportError: onnxruntime-gpu thiếu libcudnn/libcublas
    # thường ném OSError/RuntimeError -> nếu chỉ bắt ImportError thì cell crash thay vì fallback.
    importlib.invalidate_caches()
    try:
        import onnxruntime
        print('onnxruntime OK, version:', onnxruntime.__version__)
        print('Available providers:', onnxruntime.get_available_providers())
        return True
    except Exception as e:
        print(f'Chưa import được onnxruntime: {type(e).__name__}: {e}')
        return False

def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', *args], capture_output=True, text=True)
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])
    return r.returncode

if not try_import_onnxruntime():
    # onnxruntime và onnxruntime-gpu cùng chiếm package `onnxruntime`. Cài cái này đè cái kia
    # là trạng thái hỏng đã biết (mất CUDAExecutionProvider, import lỗi loạn)
    # -> luôn gỡ sạch cả hai trước khi cài lại.
    print('Gỡ sạch onnxruntime cũ rồi cài lại onnxruntime-gpu...')
    pip('uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu')
    pip('install', 'onnxruntime-gpu==1.20.0')

    if not try_import_onnxruntime():
        print('onnxruntime-gpu không cài được (khả năng chưa có wheel cho bản Python này).')
        print('Fallback sang onnxruntime bản CPU...')
        pip('uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu')
        pip('install', 'onnxruntime')
        assert try_import_onnxruntime(), 'Vẫn không cài được onnxruntime, xem log lỗi ở trên.'

In [ ]:
import cv2
import insightface
from insightface.app import FaceAnalysis

import onnxruntime
available_providers = onnxruntime.get_available_providers()
USE_CUDA = 'CUDAExecutionProvider' in available_providers
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if USE_CUDA else ['CPUExecutionProvider']
# ctx_id phải khớp với providers: 0 = GPU 0, -1 = CPU. Để nguyên 0 khi chỉ có CPU provider là mâu thuẫn.
ctx_id = 0 if USE_CUDA else -1
print('Dùng providers:', providers, '| ctx_id =', ctx_id)

app = FaceAnalysis(name='buffalo_l', providers=providers)
app.prepare(ctx_id=ctx_id, det_size=(640, 640))

swapper = insightface.model_zoo.get_model(INSWAPPER_PATH, download=False, providers=providers)

# cv2.imread trả None khi file hỏng / định dạng không hỗ trợ (heic, webp lạ...).
# Phải chặn ngay, không thì app.get(None) ném lỗi cv2 không nói gì về nguyên nhân thật.
source_img = cv2.imread(source_face_path)
assert source_img is not None, (
    f'Không đọc được ảnh {source_face_path}. Lưu lại thành .jpg/.png rồi upload lại.'
)

faces = app.get(source_img)
assert len(faces) > 0, 'Không tìm thấy khuôn mặt trong ảnh nguồn, thử ảnh khác rõ mặt hơn.'
if len(faces) > 1:
    # Thứ tự app.get() trả về KHÔNG xác định -> không được lấy faces[0].
    # Ảnh nguồn có nhiều mặt thì lấy mặt to nhất (chủ thể của ảnh).
    print(f'  (ảnh nguồn có {len(faces)} mặt, dùng mặt lớn nhất)')
source_face = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))

print('Đã detect khuôn mặt nguồn thành công.')

## 5. Làm nét mặt (GFPGAN) — cả mục này tùy thuộc `USE_GFPGAN`

Nếu bạn đặt `USE_GFPGAN = False` ở mục 0 thì **cứ chạy tuần tự cả 3 cell dưới**, chúng sẽ tự
in một dòng rồi bỏ qua. Không cần nhớ bỏ cell nào.

### Fix riêng cho `basicsr` (bug với `torchvision` mới trên Colab)

Bản `torchvision` mới đã xoá hẳn module `torchvision.transforms.functional_tensor`, trong khi
`basicsr` vẫn import `rgb_to_grayscale` theo đường cũ đó → `ModuleNotFoundError`. Hàm này nay
nằm ở `torchvision.transforms.functional`.

Cell dưới định vị package bằng `importlib.util.find_spec` — hàm này chỉ **tìm** package chứ
không chạy `__init__.py`, nên không dính đúng cái lỗi import mà ta đang muốn vá — rồi sửa mọi
file `.py` còn tham chiếu module cũ (trong cả `basicsr`, `facexlib`, `gfpgan`).

Nó cũng tự **xoá các module hỏng khỏi `sys.modules`** cả trước lẫn sau khi vá, nhờ vậy
**không cần Runtime > Restart session**.

In [ ]:
import importlib, importlib.util, sys, pathlib

OLD_MOD = 'torchvision.transforms.functional_tensor'
NEW_MOD = 'torchvision.transforms.functional'
PKGS = ('basicsr', 'facexlib', 'gfpgan')


def purge_modules():
    """Xoá các module (có thể đã import hỏng) khỏi cache.

    Phải chạy TRƯỚC find_spec: một lần import hỏng trước đó có thể để lại
    sys.modules['basicsr'] với __spec__ = None, khiến find_spec ném ValueError
    chứ không trả về None -> cell báo nhầm "chưa cài" dù basicsr đang có trên đĩa.
    """
    for mod in [m for m in list(sys.modules) if m.split('.')[0] in PKGS]:
        del sys.modules[mod]
    importlib.invalidate_caches()


def package_dir(name):
    """Trả về thư mục package mà KHÔNG import nó.

    find_spec() chỉ định vị package, không chạy __init__.py -> không dính đúng cái
    ModuleNotFoundError mà ta đang muốn vá.
    """
    try:
        spec = importlib.util.find_spec(name)
    except Exception as e:
        print(f'  (không tra được {name}: {type(e).__name__}: {e})')
        return None
    if spec is None or not spec.submodule_search_locations:
        return None
    return pathlib.Path(list(spec.submodule_search_locations)[0])


def patch_torchvision_refs():
    purge_modules()

    targets = {}
    for name in PKGS:
        d = package_dir(name)
        if d is None:
            print(f'{name:9s}: CHƯA CÀI')
        else:
            print(f'{name:9s}: {d}')
            targets[name] = d

    assert 'basicsr' in targets, (
        'Không tìm thấy basicsr. Chạy lại cell cài basicsr ở mục 1 rồi chạy lại cell này.'
    )

    patched = []
    for name, d in targets.items():
        for p in d.rglob('*.py'):
            try:
                text = p.read_text(encoding='utf-8')
            except (UnicodeDecodeError, OSError):
                continue
            if OLD_MOD not in text:
                continue
            p.write_text(text.replace(OLD_MOD, NEW_MOD), encoding='utf-8')
            patched.append(p)

    print()
    if patched:
        for p in patched:
            print(f'đã vá: {p}')
    else:
        print('Không file nào cần vá (đã vá trước đó, hoặc torchvision bản này vẫn còn functional_tensor).')

    # Purge lần nữa sau khi vá, để lần import sau đọc lại file mới trên đĩa.
    purge_modules()

    # Kiểm chứng ngay tại đây thay vì để tới cell import gfpgan mới biết.
    try:
        import basicsr.data.degradations
        print()
        print('OK: import basicsr.data.degradations thành công.')
    except Exception as e:
        print()
        print(f'VẪN LỖI: {type(e).__name__}: {e}')
        print('Nếu lỗi vẫn liên quan tới torchvision, thử Runtime > Restart session rồi chạy lại.')
        raise


if USE_GFPGAN:
    patch_torchvision_refs()
else:
    print('USE_GFPGAN = False -> bỏ qua (không có basicsr để vá).')

In [ ]:
import subprocess, sys, importlib


def try_import_gfpgan():
    importlib.invalidate_caches()
    try:
        import gfpgan
        print('gfpgan OK, version:', getattr(gfpgan, '__version__', 'unknown'))
        return True
    except Exception as e:
        # Bắt Exception: gfpgan hỏng vì basicsr/torchvision thường ném ModuleNotFoundError,
        # nhưng tuỳ phiên bản torch cũng có thể là AttributeError/OSError.
        print(f'Chưa import được gfpgan: {type(e).__name__}: {e}')
        return False


GFPGAN_AVAILABLE = False

if not USE_GFPGAN:
    print('USE_GFPGAN = False -> bỏ qua kiểm tra gfpgan.')
else:
    GFPGAN_AVAILABLE = try_import_gfpgan()

    if not GFPGAN_AVAILABLE:
        print('Thử cài lại gfpgan với log đầy đủ...')
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', 'gfpgan'],
                           capture_output=True, text=True)
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
        GFPGAN_AVAILABLE = try_import_gfpgan()

    if not GFPGAN_AVAILABLE:
        print()
        print('gfpgan không cài được -> sẽ tự động chạy tiếp mà KHÔNG làm nét.')
        print('Pipeline chính (swap mặt) vẫn chạy bình thường.')

In [ ]:
# restorer = None nghĩa là "không làm nét". Vòng lặp ở mục 6 chỉ nhìn biến này,
# nên không cần kiểm tra USE_GFPGAN lần nữa ở trong đó.
restorer = None

if USE_GFPGAN and GFPGAN_AVAILABLE:
    from gfpgan import GFPGANer
    restorer = GFPGANer(
        model_path=GFPGAN_PATH,
        upscale=1,
        arch='clean',
        channel_multiplier=2,
        bg_upsampler=None
    )
    print('GFPGAN sẵn sàng (chỉ chạy trên vùng crop quanh mặt đã swap).')
elif USE_GFPGAN:
    print('USE_GFPGAN = True nhưng gfpgan không dùng được -> chạy tiếp, chỉ swap không làm nét.')
else:
    print('USE_GFPGAN = False -> chỉ swap, không làm nét.')

## 5b. Thay tóc — segment + warp + blend

Cả mục này phụ thuộc `USE_HAIR`. Tắt thì hai cell dưới in một dòng rồi bỏ qua.

### Chuẩn bị 1 lần từ ảnh nguồn

MediaPipe chạy **một lần** trên ảnh nguồn, lấy mask xác suất lớp `hair`. Sau đó:

1. Ngưỡng `HAIR_THRESH` → **giữ lại duy nhất vùng liên thông lớn nhất**. Mask thô luôn có vài
   đốm rời (mi mắt, bóng tối, hoa văn áo bị nhận nhầm là tóc); dán chúng lên video sẽ thành
   những vệt đen bay lơ lửng cạnh đầu.
2. `MORPH_CLOSE` bịt các lỗ nhỏ trong lòng mảng tóc.
3. **Crop về đúng hộp bao quanh tóc** rồi mới lưu. Nhờ vậy mỗi frame chỉ warp một ảnh nhỏ cỡ
   cái đầu, không phải warp cả ảnh nguồn 4000×3000.
4. Làm mềm mép bằng Gaussian (`HAIR_FEATHER` × bề ngang đầu) → alpha, không phải mask 0/1.
   Mask nhị phân dán lên video cho ra đường viền răng cưa nhìn thấy ngay.
5. Đo **độ sáng trung bình của da mặt** trong ảnh nguồn (lớp `face-skin`, kênh L của LAB) — mốc
   để hoà sáng ở bước blend.

Cell cũng **cảnh báo nếu tóc bị cắt ở mép ảnh nguồn**: ảnh crop sát đầu sẽ cho mảng tóc có mép
thẳng tắp, dán lên video thành một đường cắt ngang rất lộ. Không sửa được bằng tham số — phải
đổi ảnh nguồn rộng hơn.

### Mỗi frame

**Warp.** Khớp 5 điểm mốc ảnh-nguồn → 5 điểm mốc frame bằng **Umeyama** (công thức đóng cho
phép đồng dạng: xoay + phóng đều + tịnh tiến). Vì sao không dùng `cv2.estimateAffinePartial2D`:
nó chạy RANSAC/LMEDS, với đúng 5 điểm thì kết quả **không tất định** giữa các frame gần giống
nhau → tóc giật lăn tăn không rõ nguyên nhân. Umeyama là nghiệm bình phương tối thiểu duy nhất,
cùng input cho cùng output. Có chặn nghiệm lật gương (`det < 0`) để tóc không bị soi ngược.

Tóc gắn cứng vào hộp sọ nên phép đồng dạng là xấp xỉ bậc nhất hợp lý. Nó **không** mô tả được
đầu quay ra khỏi mặt phẳng ảnh — đó là việc của cái cổng mờ dần bên dưới.

**Làm mượt.** EMA trên **6 tham số của ma trận biến đổi**, không phải trên ảnh đã warp: làm mượt
tham số thì tóc di chuyển liền mạch mà vẫn nét; làm mượt ảnh thì được cái nhoè. Mất mặt vài
frame (che, quay lưng) thì **xoá trạng thái EMA**, nếu không tóc sẽ trượt từ vị trí cũ sang vị
trí mới như bay.

**Cổng theo độ quay đầu.** Đo độ lệch của mũi so với trung điểm hai mắt, chiếu lên **trục hai
mắt** (nên đã tính cả nghiêng đầu roll), chuẩn hoá theo khoảng cách hai mắt. `0` = chính diện.
Vượt `HAIR_MAX_TURN[0]` thì alpha giảm dần, tới `[1]` thì tắt hẳn — thà giữ tóc gốc còn hơn dán
một mảng tóc chính diện lên cái đầu đang nhìn ngang.

**Mask bảo vệ mặt.** Segment frame (một lần, trên vùng đầu resize 256×256) lấy `hair` +
`face-skin`. Mask bảo vệ = `face-skin` × một **dốc mềm vuông góc trục hai mắt** — chứ không phải
dốc theo trục y của ảnh, vì đầu nghiêng thì dốc theo y sẽ cắt chéo qua mặt. `HAIR_COVER_FOREHEAD`
dịch dốc này lên/xuống: tóc mái phủ trán được, nhưng chặn cứng trước khi tới mắt.

**Vùng làm việc (`HAIR_ROI`).** Mọi thao tác trên chỉ chạy trong một hộp quanh đầu, tính theo
khoảng cách hai mắt — không phải trên cả khung hình. Hộp này phải phủ **cả tóc cũ**, không chỉ
cái đầu: phần tóc cũ nằm ngoài hộp thì bước xoá tóc thừa không nhìn thấy, nên vẫn lộ trong video.
Người trong video tóc dài ngang lưng thì tăng số thứ ba (`4.5` → `6.5`).

**Xoá tóc cũ thò ra.** `tóc-cũ AND NOT tóc-mới` = phần lộ ra khi tóc nguồn ngắn hơn.
`cv2.inpaint` (Telea) trên bản thu nhỏ 1/2 của vùng đầu, rồi blend mép mềm. Đây là chỗ tốn nhất
của phần tóc và chỉ chạy khi thật sự có phần thừa. Hạn chế đã biết: inpaint từng frame độc lập
nên vùng nền vá được có thể lăn tăn giữa các frame.

**Hoà sáng.** Không dùng `cv2.seamlessClone`: Poisson blending sẽ kéo màu vùng dán về theo nền
xung quanh, tức là **xoá luôn màu tóc của ảnh nguồn** — đúng cái ta muốn giữ. Thay vào đó chỉ
cộng bù kênh L: `dL = L(da mặt trong frame) − L(da mặt ảnh nguồn)`, nhân `HAIR_LIGHT_MATCH`.
Da mặt là mốc tốt vì luôn có mặt trong cả hai ảnh, và **sau khi swap thì da mặt trong frame đã
được inswapper hoà theo ánh sáng video rồi** — nên nó chính là "ánh sáng của cảnh này" đo trên
đúng khuôn mặt đó.

In [ ]:
import numpy as np
import cv2

# Nhãn của model selfie_multiclass_256x256 (theo tài liệu MediaPipe).
LBL_BACKGROUND, LBL_HAIR, LBL_BODY_SKIN, LBL_FACE_SKIN, LBL_CLOTHES, LBL_OTHERS = range(6)

SEG_SIZE = 256   # model vốn chạy ở 256x256; đưa vào đúng cỡ này để nó không phải nội suy 2 lần


def mask_2d(arr):
    """Ép mask của MediaPipe về đúng 2D (H, W).

    Có bản mediapipe trả confidence mask shape (H, W, 1) thay vì (H, W). Trộn hai kiểu này
    lại là lỗi ngầm rất khó đoán: cv2.GaussianBlur/resize LẶNG LẼ bỏ chiều cuối, nên một mask
    thành 2D còn mask kia vẫn 3D, rồi phép nhân giữa chúng nổ
    "non-broadcastable output operand ... doesn't match the broadcast shape (256,256,256)"
    - hoặc tệ hơn, np.nonzero() trả 3 mảng thay vì 2. Chuẩn hoá ngay tại nguồn, một lần.
    """
    # copy=True là BẮT BUỘC, không phải cho chắc: np.asarray() trên mảng float32 đã liền khối
    # trả về CHÍNH mảng đó, tức là một view vào bộ nhớ MediaPipe - và bộ nhớ đó bị ghi đè ở lần
    # segment() sau, làm mask của frame trước đổi giá trị sau lưng mình (EMA thành vô nghĩa).
    a = np.array(arr, dtype=np.float32, copy=True)
    if a.ndim == 3 and a.shape[2] == 1:
        a = a[:, :, 0]              # view vào BẢN COPY ở trên -> vẫn an toàn
    assert a.ndim == 2, f'Mask của MediaPipe có shape lạ: {a.shape}'
    return np.ascontiguousarray(a)


def umeyama_similarity(src, dst):
    """Phép đồng dạng (xoay + phóng đều + tịnh tiến) khớp src -> dst, trả ma trận 2x3.

    Công thức đóng của Umeyama, KHÔNG dùng cv2.estimateAffinePartial2D: estimator đó chạy
    RANSAC/LMEDS nên với đúng 5 điểm mốc, hai frame gần như giống nhau vẫn có thể cho hai ma
    trận khác nhau -> tóc giật lăn tăn. Ở đây cùng input luôn cho cùng output.
    """
    src = np.asarray(src, dtype=np.float64)
    dst = np.asarray(dst, dtype=np.float64)
    n = len(src)
    m_s, m_d = src.mean(0), dst.mean(0)
    s, d = src - m_s, dst - m_d

    C = (d.T @ s) / n
    U, S, Vt = np.linalg.svd(C)

    D = np.eye(2)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        # Chặn nghiệm phản chiếu: không có nó, landmark nhiễu ở frame gần nghiêng hẳn có thể
        # cho ra ma trận lật gương -> tóc bị soi ngược trong một vài frame.
        D[1, 1] = -1.0

    R = U @ D @ Vt
    var_s = (s ** 2).sum() / n
    scale = float((S * np.diag(D)).sum() / max(var_s, 1e-12))

    M = np.zeros((2, 3), dtype=np.float64)
    M[:, :2] = scale * R
    M[:, 2] = m_d - scale * R @ m_s
    return M


def least_squares_affine(src, dst):
    """Affine 6 bậc tự do bằng bình phương tối thiểu (cũng tất định như trên).

    Bám được cả phần co ngang khi đầu quay, nhưng nhạy với nhiễu landmark hơn đồng dạng.
    """
    src = np.asarray(src, dtype=np.float64)
    dst = np.asarray(dst, dtype=np.float64)
    A = np.hstack([src, np.ones((len(src), 1))])
    sol, *_ = np.linalg.lstsq(A, dst, rcond=None)     # (3, 2)
    return sol.T.copy()                                # (2, 3)


def frontality(kps):
    """Độ lệch của mũi so với trung điểm hai mắt, chiếu lên trục hai mắt / khoảng cách hai mắt.

    0 = chính diện, càng lớn = càng quay đầu. Chiếu lên TRỤC HAI MẮT chứ không lấy hiệu toạ độ
    x, nhờ vậy đầu nghiêng (roll) không bị tính lẫn thành quay (yaw).
    """
    le, re, nose = kps[0], kps[1], kps[2]
    eye = re - le
    d = float(np.linalg.norm(eye))
    if d < 1e-6:
        return 0.0
    u = eye / d
    mid = (le + re) / 2.0
    return float(np.dot(nose - mid, u) / d)


def largest_blob(mask_u8):
    """Giữ lại duy nhất vùng liên thông lớn nhất.

    Mask tóc thô luôn kèm vài đốm rời: mi mắt, vùng tối dưới cổ, hoa văn áo bị nhận nhầm.
    Không lọc thì chúng thành những vệt đen bay lơ lửng cạnh đầu trong video.
    """
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    if n <= 1:
        return mask_u8
    biggest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return (labels == biggest).astype(np.uint8)


class HairTransfer:
    """Chuyển mái tóc từ ảnh nguồn sang từng frame video: segment -> warp -> blend."""

    def __init__(self, model_path, source_bgr, source_kps):
        import mediapipe as mp
        from mediapipe.tasks import python as mp_python
        from mediapipe.tasks.python import vision as mp_vision

        self._mp = mp
        common = dict(base_options=mp_python.BaseOptions(model_asset_path=model_path),
                      running_mode=mp_vision.RunningMode.IMAGE)
        try:
            options = mp_vision.ImageSegmenterOptions(
                **common,
                output_category_mask=False,   # chỉ cần confidence mask (float, mép mềm)
                output_confidence_masks=True,
            )
        except TypeError as e:
            # mediapipe < 0.10.3 dùng bộ tham số khác (output_type/activation). Không ghim
            # version ở mục 1 (dễ hết wheel cho Python mới trên Colab) nên chấp nhận cả hai:
            # để mặc định, rồi _segment() tự xoay theo cái mà result thực sự trả về.
            print(f'  (ImageSegmenterOptions bản cũ: {e})')
            options = mp_vision.ImageSegmenterOptions(**common)
        self.segmenter = mp_vision.ImageSegmenter.create_from_options(options)

        # Trạng thái theo thời gian (EMA). None = chưa có frame trước / vừa mất mặt.
        self.prev_M = None
        self.prev_masks = None
        self.n_faded = 0
        self.n_inpainted = 0

        self._build_source(source_bgr, np.asarray(source_kps, dtype=np.float64))

    # ---------------------------------------------------------------- segment
    def _segment(self, bgr):
        """Trả (hair, face_skin) - hai mask xác suất float32 cỡ SEG_SIZE x SEG_SIZE.

        Luôn đưa vào đúng SEG_SIZE: model chạy ở 256x256 rồi tự phóng mask về cỡ ảnh đầu vào,
        nên đưa ảnh to vào chỉ tốn thêm một lần nội suy của MediaPipe mà không thêm thông tin.
        Tự phóng mask về cỡ thật bằng cv2 rẻ hơn nhiều.
        """
        small = cv2.resize(bgr, (SEG_SIZE, SEG_SIZE), interpolation=cv2.INTER_AREA)
        rgb = np.ascontiguousarray(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))
        mp_img = self._mp.Image(image_format=self._mp.ImageFormat.SRGB, data=rgb)
        res = self.segmenter.segment(mp_img)
        # copy=True: numpy_view() chỉ là view vào bộ nhớ của MediaPipe, bị ghi đè ở lần
        # segment() sau -> không copy thì mask của frame trước đổi giá trị sau lưng mình.
        masks = getattr(res, 'confidence_masks', None)
        if masks:
            # mask_2d() vừa copy vừa ép về 2D - xem docstring của nó để biết vì sao cần cả hai.
            hair = mask_2d(masks[LBL_HAIR].numpy_view())
            face = mask_2d(masks[LBL_FACE_SKIN].numpy_view())
        else:
            # Bản mediapipe không cho confidence mask -> lấy category mask (0/1, mép gắt).
            # Vẫn chạy được vì bước sau còn làm mềm mép bằng Gaussian.
            cat = mask_2d(res.category_mask.numpy_view())
            hair = (cat == LBL_HAIR).astype(np.float32)
            face = (cat == LBL_FACE_SKIN).astype(np.float32)
        return hair, face

    # ------------------------------------------------------- chuẩn bị 1 lần
    def _build_source(self, img, kps):
        h, w = img.shape[:2]
        hair_s, face_s = self._segment(img)
        hair = cv2.resize(hair_s, (w, h), interpolation=cv2.INTER_LINEAR)
        face = cv2.resize(face_s, (w, h), interpolation=cv2.INTER_LINEAR)

        eye_dist = float(np.linalg.norm(kps[1] - kps[0]))
        head_w = 2.2 * eye_dist          # bề ngang đầu ~ 2.2 lần khoảng cách hai mắt

        m = (hair >= HAIR_THRESH).astype(np.uint8)
        assert m.sum() >= 0.02 * head_w * head_w, (
            "Không tìm thấy vùng tóc trong ảnh nguồn (mask gần như trống).\n"
            "Thử: giảm HAIR_THRESH (0.35), hoặc dùng ảnh khác thấy rõ tóc - ảnh đội mũ, "
            "crop sát mặt, hoặc nền cùng màu tóc đều dễ hỏng bước này."
        )
        m = largest_blob(m)
        k = max(3, (int(0.03 * head_w) | 1))
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((k, k), np.uint8))

        # Cảnh báo tóc bị cắt ở mép ảnh: mảng tóc sẽ có mép thẳng, dán lên video rất lộ.
        b = int(max(2, 0.005 * max(h, w)))
        edge = max(m[:b].mean(), m[-b:].mean(), m[:, :b].mean(), m[:, -b:].mean())
        if edge > 0.25:
            print("  CANH BAO: tóc trong ảnh nguồn bị cắt ở mép ảnh -> mảng tóc dán lên video")
            print("            sẽ có một đường cắt thẳng rất lộ. Nên dùng ảnh chụp rộng hơn,")
            print("            thấy hết mái tóc. Không tham số nào chữa được chỗ này.")

        ys, xs = np.nonzero(m)
        pad = int(0.04 * head_w) + 2
        x1 = max(0, int(xs.min()) - pad); x2 = min(w, int(xs.max()) + 1 + pad)
        y1 = max(0, int(ys.min()) - pad); y2 = min(h, int(ys.max()) + 1 + pad)

        # Crop về hộp bao quanh tóc: mỗi frame chỉ warp một ảnh cỡ cái đầu, không phải cả ảnh gốc.
        self.src_bgr = np.ascontiguousarray(img[y1:y2, x1:x2])
        sigma = max(1.0, HAIR_FEATHER * head_w)
        self.src_alpha = cv2.GaussianBlur(m[y1:y2, x1:x2].astype(np.float32), (0, 0), sigmaX=sigma)
        self.src_kps = kps - np.array([x1, y1], dtype=np.float64)

        # Mốc ánh sáng: độ sáng trung bình của da mặt trong ảnh nguồn (kênh L của LAB).
        wsum = float(face.sum())
        L = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)[..., 0].astype(np.float32)
        self.src_face_L = float((L * face).sum() / wsum) if wsum > 50 else float(L.mean())

        print(f'  tóc nguồn: {int(m.sum())} px, crop {x2 - x1}x{y2 - y1} '
              f'(ảnh gốc {w}x{h}), khoảng cách hai mắt {eye_dist:.0f} px')
        print(f'  độ sáng da mặt ảnh nguồn (L): {self.src_face_L:.1f}/255')

    # ---------------------------------------------------------------- warp
    def _transform(self, kps):
        M = (least_squares_affine(self.src_kps, kps) if HAIR_WARP_MODE == 'affine'
             else umeyama_similarity(self.src_kps, kps))
        if self.prev_M is not None and HAIR_SMOOTH > 0:
            # EMA trên 6 THAM SỐ của ma trận, không phải trên ảnh đã warp: mượt tham số thì
            # tóc di chuyển liền mạch mà vẫn nét; mượt ảnh thì chỉ được cái nhoè.
            M = HAIR_SMOOTH * self.prev_M + (1.0 - HAIR_SMOOTH) * M
        self.prev_M = M
        return M

    def _pose_gate(self, kps):
        r = abs(frontality(kps))
        t0, t1 = HAIR_MAX_TURN
        if r <= t0:
            return 1.0
        if r >= t1:
            return 0.0
        return float((t1 - r) / max(t1 - t0, 1e-6))

    def reset(self):
        """Xoá trạng thái thời gian. Gọi khi frame không detect được mặt.

        Không reset thì frame có mặt tiếp theo sẽ EMA với ma trận của lần thấy mặt cuối
        (có thể cách đó 2 giây) -> tóc trượt vào khung như bay.
        """
        self.prev_M = None
        self.prev_masks = None

    # ---------------------------------------------------------------- blend
    def apply(self, frame, kps):
        """Dán tóc nguồn lên frame (ghi trực tiếp vào frame). Trả (frame, đã_dán?)."""
        kps = np.asarray(kps, dtype=np.float64)
        H, W = frame.shape[:2]

        gate = HAIR_STRENGTH * self._pose_gate(kps)
        if gate <= 0.02:
            self.n_faded += 1
            self.prev_masks = None
            self._transform(kps)          # vẫn cập nhật EMA để lúc quay lại mặt không bị nhảy
            return frame, False

        M = self._transform(kps)
        sh, sw = self.src_alpha.shape

        # ROI = hộp tóc mới ĐÃ warp  hợp với  hộp quanh đầu (cần cả tóc CŨ để xoá phần thừa).
        corners = np.array([[0, 0], [sw, 0], [sw, sh], [0, sh]], dtype=np.float64)
        proj = corners @ M[:, :2].T + M[:, 2]
        d = max(float(np.linalg.norm(kps[1] - kps[0])), 1.0)
        cx, cy = kps.mean(0)
        # Hộp quanh đầu phải phủ được cả TÓC CŨ, không chỉ cái đầu: phần tóc cũ nằm ngoài hộp
        # này thì bước xoá tóc thừa không nhìn thấy -> vẫn lộ trong video. Đó là lý do bề
        # xuống dưới (HAIR_ROI[2]) là tham số, không phải hằng số: tóc dài ngang lưng cần 6-7.
        side, up, down = HAIR_ROI
        head = np.array([[cx - side * d, cy - up * d], [cx + side * d, cy + down * d]])
        pts = np.vstack([proj, head])

        x1 = max(0, int(np.floor(pts[:, 0].min()))); x2 = min(W, int(np.ceil(pts[:, 0].max())))
        y1 = max(0, int(np.floor(pts[:, 1].min()))); y2 = min(H, int(np.ceil(pts[:, 1].max())))
        rw, rh = x2 - x1, y2 - y1
        if rw < 16 or rh < 16:
            return frame, False

        # Warp CHỈ vào ROI: cộng bù tịnh tiến vào ma trận, thay vì warp ra cả khung rồi cắt.
        M_roi = M.copy()
        M_roi[0, 2] -= x1
        M_roi[1, 2] -= y1
        warped = cv2.warpAffine(self.src_bgr, M_roi, (rw, rh), flags=cv2.INTER_LINEAR,
                                borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
        w_alpha = cv2.warpAffine(self.src_alpha, M_roi, (rw, rh), flags=cv2.INTER_LINEAR,
                                 borderMode=cv2.BORDER_CONSTANT, borderValue=0)

        roi = frame[y1:y2, x1:x2]

        # --- segment frame (1 lần, dùng cho cả 3 việc: bảo vệ mặt, xoá tóc cũ, đo ánh sáng) ---
        hair_s, face_s = self._segment(roi)
        if self.prev_masks is not None and HAIR_MASK_SMOOTH > 0:
            # EMA ngay trong không gian 256x256 đã chuẩn hoá theo ROI, trước khi phóng to:
            # ROI đi theo ma trận đã làm mượt nên nó dịch chuyển êm, hai mask liên tiếp gần
            # như trùng khớp. Chuyển động rất nhanh thì chỗ này hơi nhoè - đó là cái giá.
            hair_s = HAIR_MASK_SMOOTH * self.prev_masks[0] + (1 - HAIR_MASK_SMOOTH) * hair_s
            face_s = HAIR_MASK_SMOOTH * self.prev_masks[1] + (1 - HAIR_MASK_SMOOTH) * face_s
        self.prev_masks = (hair_s, face_s)

        hair_t = cv2.resize(hair_s, (rw, rh), interpolation=cv2.INTER_LINEAR)
        face_t = cv2.resize(face_s, (rw, rh), interpolation=cv2.INTER_LINEAR)

        # --- mask bảo vệ mặt: dốc mềm VUÔNG GÓC TRỤC HAI MẮT, không phải theo trục y của ảnh ---
        le, re = kps[0] - [x1, y1], kps[1] - [x1, y1]
        mouth_mid = (kps[3] + kps[4]) / 2.0 - [x1, y1]
        mid = (le + re) / 2.0
        eye = re - le
        nrm = np.array([-eye[1], eye[0]], dtype=np.float64)
        nrm /= max(np.linalg.norm(nrm), 1e-6)
        if np.dot(nrm, mouth_mid - mid) < 0:
            nrm = -nrm                     # cho vector luôn chỉ xuống phía miệng
        # s = khoảng cách xuống dưới đường hai mắt, theo đơn vị "khoảng cách hai mắt".
        gx = np.arange(rw, dtype=np.float32)
        gy = np.arange(rh, dtype=np.float32)[:, None]
        s = ((gx - mid[0]) * nrm[0] + (gy - mid[1]) * nrm[1]) / d
        s0 = -0.80 + 0.70 * float(np.clip(HAIR_COVER_FOREHEAD, 0.0, 1.0))
        protect = face_t * np.clip((s - s0) / 0.25, 0.0, 1.0)

        alpha = np.clip(w_alpha, 0.0, 1.0) * (1.0 - protect) * gate

        # --- xoá tóc cũ thò ra ngoài vùng tóc mới ---
        base = roi
        if HAIR_REMOVE_LEFTOVER:
            leftover = ((hair_t > 0.5) & (alpha < 0.35)).astype(np.uint8)
            if leftover.sum() > 0.002 * rw * rh and rw >= 32 and rh >= 32:
                sm = cv2.resize(roi, (rw // 2, rh // 2), interpolation=cv2.INTER_AREA)
                lm = cv2.resize(leftover, (rw // 2, rh // 2), interpolation=cv2.INTER_NEAREST)
                # Nới thêm chút: inpaint sát mép tóc sẽ lấy chính pixel tóc làm mẫu -> vá ra tóc.
                lm = cv2.dilate(lm, np.ones((5, 5), np.uint8))
                filled = cv2.inpaint(sm, lm, 4, cv2.INPAINT_TELEA)
                filled = cv2.resize(filled, (rw, rh), interpolation=cv2.INTER_LINEAR)
                lw = cv2.GaussianBlur(
                    cv2.dilate(leftover, np.ones((5, 5), np.uint8)).astype(np.float32),
                    (0, 0), sigmaX=max(1.5, 0.01 * d))[..., None]
                base = (roi.astype(np.float32) * (1 - lw) + filled.astype(np.float32) * lw)
                self.n_inpainted += 1

        # --- hoà sáng: chỉ bù kênh L. KHÔNG dùng seamlessClone (nó sẽ xoá màu tóc nguồn) ---
        if HAIR_LIGHT_MATCH > 0:
            fsum = float(face_t.sum())
            if fsum > 50:
                L_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2LAB)[..., 0].astype(np.float32)
                dL = (float((L_roi * face_t).sum() / fsum) - self.src_face_L) * HAIR_LIGHT_MATCH
                if abs(dL) > 1.0:
                    lab = cv2.cvtColor(warped, cv2.COLOR_BGR2LAB)
                    lab[..., 0] = np.clip(lab[..., 0].astype(np.float32) + dL, 0, 255).astype(np.uint8)
                    warped = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

        a3 = alpha[..., None]
        out = np.asarray(base, dtype=np.float32) * (1.0 - a3) + warped.astype(np.float32) * a3
        frame[y1:y2, x1:x2] = np.clip(out, 0, 255).astype(np.uint8)
        return frame, True


hair_engine = None

if not USE_HAIR:
    print('USE_HAIR = False -> giữ nguyên tóc của người trong video.')
elif not HAIR_AVAILABLE:
    print('USE_HAIR = True nhưng mediapipe không dùng được -> chạy tiếp, chỉ swap mặt.')
else:
    print('Đang chuẩn bị mẫu tóc từ ảnh nguồn...')
    hair_engine = HairTransfer(SEGMENTER_PATH, source_img, source_face.kps)
    print('Sẵn sàng thay tóc.')

In [ ]:
# Xem mask tóc lấy từ ảnh nguồn TRƯỚC khi chạy cả video.
# Mask sai ở đây thì mọi frame đều sai - phát hiện sớm rẻ hơn nhiều so với xử lý xong mới thấy.
import numpy as np
import cv2

if hair_engine is None:
    print('Không có hair_engine -> không có gì để xem.')
else:
    from google.colab.patches import cv2_imshow

    crop = hair_engine.src_bgr
    a = hair_engine.src_alpha[..., None]

    # Trái: ảnh gốc đã crop. Giữa: chỉ phần tóc (nền xám). Phải: tô đỏ đè lên để soi mép.
    only_hair = (crop * a + 128 * (1 - a)).astype(np.uint8)
    red = np.zeros_like(crop); red[..., 2] = 255
    overlay = (crop * (1 - 0.5 * a) + red * (0.5 * a)).astype(np.uint8)

    strip = np.hstack([crop, only_hair, overlay])
    scale = min(1.0, 1100 / strip.shape[1])
    if scale < 1.0:
        strip = cv2.resize(strip, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)

    print('gốc  |  chỉ tóc  |  tô đỏ vùng tóc')
    cv2_imshow(strip)
    print()
    print('Nhìn ảnh thứ 3 và tự hỏi:')
    print('  - Có ăn lẹm sang nền / vai / lông mày không?  -> tăng HAIR_THRESH')
    print('  - Có hụt mất phần tóc ngoài rìa không?        -> giảm HAIR_THRESH (0.35)')
    print('  - Tóc có bị cắt cụt ở mép ảnh không?          -> đổi ảnh nguồn chụp rộng hơn')
    print('Sửa xong thì chạy lại cell cấu hình (mục 0) rồi chạy lại cell trên.')

## 6. Xử lý video

Mỗi frame đi qua **bốn bước, theo đúng thứ tự này**:

1. `app.get(frame)` → detect mặt, `pick_face(...)` → chọn **mặt lớn nhất** (video 1 người nên
   không cần tracking danh tính)
2. `swapper.get(..., paste_back=True)` → swap mặt, dán về luôn bằng mask mặc định của inswapper
3. `enhance_face_region(...)` → làm nét, chỉ khi `restorer is not None`
4. `hair_engine.apply(...)` → **thay tóc**, chỉ khi `hair_engine is not None`

**Bước 4 phải nằm sau bước 3.** Ô crop của GFPGAN nới bbox thêm 40% nên lấn sang cả tóc; dán
tóc trước thì GFPGAN sẽ vẽ lại đúng đường ghép tóc vừa dán, mỗi frame một kiểu — nhấp nháy ngay
ở chỗ mắt người dễ nhìn thấy nhất.

Mục này chia làm **ba cell** thay vì một:

- **6.0** định nghĩa `process_frame()` — dùng chung cho cả preview lẫn vòng lặp thật, nên không
  có chuyện preview chạy một đường mà video ra một nẻo.
- **6.1** chạy **đúng một frame** và in ảnh trước/sau. Chỉnh tham số tóc ở mục 0 rồi chạy lại
  hai cell (mục 0 → 6.1) là xong một vòng thử — vài giây, thay vì chờ cả video.
- **6.2** chạy toàn bộ video, ghi thẳng vào `ffmpeg` qua pipe và mux audio trong cùng một pass.

> **Frame không detect được mặt** thì giữ nguyên khung gốc (cả mặt thật lẫn tóc thật lộ ra),
> và trạng thái làm mượt của tóc được xoá — nếu không, frame thấy mặt lại sẽ có mảng tóc trượt
> từ vị trí cũ sang vị trí mới như bay. Tỉ lệ frame này in ở cuối mục 6.2.

In [ ]:
import os, sys, subprocess, time
import numpy as np
import cv2
from tqdm import tqdm

MIN_DET_SCORE = 0.5      # bỏ qua detect yếu (thường là false positive trong nền)
GFPGAN_PAD    = 0.4      # nới bbox bao nhiêu lần khi crop để làm nét
final_output  = '/content/output_final.mp4'

# globals().get: cho phép bỏ qua HẲN mục 5 / 5b (không chạy cell nào ở đó) mà vẫn chạy được
# pipeline chính, thay vì NameError.
restorer    = globals().get('restorer', None)
hair_engine = globals().get('hair_engine', None)
print('Làm nét mặt:', 'BẬT' if restorer is not None else 'TẮT')
print('Thay tóc   :', 'BẬT' if hair_engine is not None else 'TẮT')


def pick_face(faces):
    """Chọn khuôn mặt để swap trong một frame. Trả về None nếu không có mặt nào đủ tốt.

    Video 1 người nên không cần tracking theo danh tính: mặt LỚN NHẤT là chủ thể.
    KHÔNG dùng faces[0] vì thứ tự app.get() trả về không xác định.
    """
    cands = [f for f in faces if f.det_score >= MIN_DET_SCORE]
    if not cands:
        return None
    return max(cands, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))


def enhance_face_region(img, face, pad=GFPGAN_PAD):
    """Chỉ làm nét vùng quanh khuôn mặt vừa swap, KHÔNG chạy trên cả frame.

    Gọi restorer.enhance() trên nguyên frame khiến GFPGAN chạy lại một bộ face detector
    thứ hai trên toàn khung (trùng lặp với app.get() vừa chạy), làm nét luôn cả những mặt
    trong nền không hề bị swap, và resize LANCZOS toàn frame mỗi lượt.
    """
    h, w = img.shape[:2]
    x1, y1, x2, y2 = face.bbox
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, int(x1 - pad * bw)); x2 = min(w, int(x2 + pad * bw))
    y1 = max(0, int(y1 - pad * bh)); y2 = min(h, int(y2 + pad * bh))
    cw, ch = x2 - x1, y2 - y1
    if cw < 64 or ch < 64:
        return img

    crop = img[y1:y2, x1:x2]
    _, _, restored = restorer.enhance(crop, has_aligned=False,
                                      only_center_face=True, paste_back=True)
    if restored is None:
        return img
    if restored.shape[:2] != (ch, cw):
        restored = cv2.resize(restored, (cw, ch), interpolation=cv2.INTER_LANCZOS4)

    # Blend viền mềm để vùng crop không để lại đường viền hình chữ nhật thấy rõ.
    mask = np.zeros((ch, cw), np.float32)
    cv2.rectangle(mask, (int(0.12 * cw), int(0.12 * ch)),
                  (int(0.88 * cw), int(0.88 * ch)), 1.0, -1)
    mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=max(2.0, 0.05 * max(cw, ch)))[..., None]
    img[y1:y2, x1:x2] = (restored * mask + crop * (1.0 - mask)).astype(np.uint8)
    return img


STATS = dict(frames=0, swapped=0, haired=0, detect=0.0, swap=0.0, enhance=0.0, hair=0.0)


def process_frame(frame):
    """Xử lý MỘT frame. Dùng chung cho cả cell thử 1 frame lẫn vòng lặp cả video.

    Thứ tự: detect -> swap -> làm nét -> THAY TÓC.
    Tóc phải là bước cuối: ô crop của GFPGAN (pad 0.4) lấn sang cả tóc, dán tóc trước thì
    GFPGAN sẽ vẽ lại chính đường ghép tóc, mỗi frame một kiểu -> nhấp nháy.
    """
    out = frame

    t0 = time.perf_counter()
    target_face = pick_face(app.get(frame))
    t1 = time.perf_counter()
    STATS['detect'] += t1 - t0

    if target_face is None:
        # Mất mặt -> xoá trạng thái thời gian, không thì lúc thấy lại tóc sẽ trượt vào khung.
        if hair_engine is not None:
            hair_engine.reset()
        STATS['frames'] += 1
        return out

    # swapper.get(paste_back=True) trả về mảng MỚI, nên các bước sau ghi đè tại chỗ đều an toàn.
    out = swapper.get(frame, target_face, source_face, paste_back=True)
    t2 = time.perf_counter()
    STATS['swap'] += t2 - t1

    if restorer is not None:
        out = enhance_face_region(out, target_face)
    t3 = time.perf_counter()
    STATS['enhance'] += t3 - t2

    if hair_engine is not None:
        out, applied = hair_engine.apply(out, target_face.kps)
        STATS['haired'] += int(applied)
    STATS['hair'] += time.perf_counter() - t3

    STATS['swapped'] += 1
    STATS['frames'] += 1
    return out


def reset_state():
    """Đưa STATS và trạng thái thời gian của tóc về 0 (gọi trước mỗi lần chạy lại)."""
    for k in STATS:
        STATS[k] = 0 if isinstance(STATS[k], int) else 0.0
    if hair_engine is not None:
        hair_engine.reset()
        hair_engine.n_faded = 0
        hair_engine.n_inpainted = 0


print('Đã định nghĩa process_frame(). Chạy cell 6.1 để thử 1 frame trước khi làm cả video.')

In [ ]:
# Thử ĐÚNG MỘT frame để soi kết quả và chỉnh tham số. Vài giây một vòng, thay vì xử lý cả video.
# Chỉnh tham số ở mục 0 -> chạy lại cell mục 0 -> chạy lại cell này. Không cần chạy lại gì khác.
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

PREVIEW_AT = 0.5      # lấy frame ở đâu trong video: 0.0 = đầu, 0.5 = giữa, 0.95 = gần cuối

cap = cv2.VideoCapture(source_video_path)
assert cap.isOpened(), f'Không mở được video: {source_video_path}'
n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if n_total > 0:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(n_total * PREVIEW_AT))
ok, frame = cap.read()
if not ok:
    # Seek thất bại với một số container -> quay lại đọc tuần tự từ đầu.
    cap.release()
    cap = cv2.VideoCapture(source_video_path)
    ok, frame = cap.read()
cap.release()
assert ok, 'Không đọc được frame nào từ video.'

before = frame.copy()

# Kiểm tra trước khi xử lý: frame preview không có mặt thì mọi thứ bên dưới đều vô nghĩa
# (và phần in số liệu sẽ nổ AttributeError trên None). Đổi PREVIEW_AT rồi chạy lại.
preview_face = pick_face(app.get(before))
assert preview_face is not None, (
    f'Không detect được mặt ở frame {PREVIEW_AT:.0%} của video. '
    'Đổi PREVIEW_AT sang mốc khác (0.1 / 0.3 / 0.7) rồi chạy lại cell này.'
)

reset_state()
# Chạy 2 lần trên cùng 1 frame: lần đầu khởi tạo EMA (prev_M = None), lần hai mới đúng trạng
# thái mà video thật sẽ có. Không làm vậy thì preview trông khác kết quả cuối một chút.
process_frame(frame.copy())
after = process_frame(frame.copy())

strip = np.hstack([before, after])
scale = min(1.0, 1100 / strip.shape[1])
if scale < 1.0:
    strip = cv2.resize(strip, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
print('TRƯỚC  |  SAU')
cv2_imshow(strip)

if hair_engine is not None:
    r = abs(frontality(np.asarray(preview_face.kps, dtype=np.float64)))
    t0, t1 = HAIR_MAX_TURN
    print()
    print(f'Độ quay đầu của frame này: {r:.2f}  (mờ tóc từ {t0}, tắt hẳn từ {t1})')
    print(f'Đã dán tóc: {"CÓ" if STATS["haired"] else "KHÔNG"}')
    print()
    print('Thấy gì thì chỉnh nấy (ở mục 0):')
    print('  tóc che mất trán/mắt          -> giảm HAIR_COVER_FOREHEAD')
    print('  còn thấy tóc cũ thò ra hai bên -> bật HAIR_REMOVE_LEFTOVER = True')
    print('  tóc cũ vẫn thò ra ở xa (dưới vai) -> tăng HAIR_ROI (số thứ 3)')
    print('  tóc quá sáng / quá tối so cảnh -> chỉnh HAIR_LIGHT_MATCH')
    print('  mép tóc gắt, thấy rõ đường ghép -> tăng HAIR_FEATHER')
    print('  tóc lệch khỏi đầu              -> thử HAIR_WARP_MODE = "affine"')
    print('  tóc biến mất dù mặt gần chính diện -> nới HAIR_MAX_TURN')

In [ ]:
import os, subprocess, time
import numpy as np
import cv2
from tqdm import tqdm

cap = cv2.VideoCapture(source_video_path)
assert cap.isOpened(), f'Không mở được video: {source_video_path}'

fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or fps != fps or fps <= 0:       # 0.0 hoặc NaN với một số container
    print('Không đọc được fps từ video, mặc định 25.')
    fps = 25.0

# CAP_PROP_FRAME_COUNT chỉ là ước lượng (sai với VFR / mp4 thiếu index) -> chỉ dùng cho progress bar.
# Vòng lặp đọc tới khi hết frame thật sự, thay vì range(total_frames) (đếm thiếu = cụt đuôi video).
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None

# Lấy kích thước từ frame THẬT, không từ CAP_PROP_FRAME_WIDTH/HEIGHT: video quay dọc có metadata
# rotation làm hai giá trị đó bị hoán đổi so với frame OpenCV trả về -> ffmpeg nhận rawvideo sai size.
ret, frame = cap.read()
assert ret, 'Không đọc được frame nào từ video.'
height, width = frame.shape[:2]

print(f'Video: {width}x{height} @ {fps:.2f}fps, ~{total_frames} frames')

# Ghi frame thô thẳng vào ffmpeg và ghép audio ngay trong cùng một pass.
ffmpeg_cmd = [
    'ffmpeg', '-y', '-loglevel', 'error',
    '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', f'{width}x{height}', '-r', f'{fps}', '-i', 'pipe:0',
    '-i', source_video_path,
    '-map', '0:v:0', '-map', '1:a:0?',        # '?' = không có audio thì bỏ qua, không lỗi
    '-c:v', 'libx264', '-crf', '18', '-preset', 'fast',
    '-pix_fmt', 'yuv420p',                    # để trình duyệt/IPython.display.Video phát được
    '-c:a', 'aac', '-shortest',
    final_output,
]
proc = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE)

reset_state()                                  # xoá số liệu + trạng thái EMA của cell preview
pbar = tqdm(total=total_frames, unit='frame')
t_start = time.perf_counter()
rc = None
try:
    while frame is not None:
        result_frame = process_frame(frame)

        proc.stdin.write(np.ascontiguousarray(result_frame).tobytes())
        pbar.update(1)

        ret, frame = cap.read()
        if not ret:
            frame = None
finally:
    # Không release trong finally thì khi bấm Stop giữa chừng, ffmpeg treo và file mp4 hỏng.
    pbar.close()
    cap.release()
    try:
        proc.stdin.close()
    except BrokenPipeError:
        pass
    rc = proc.wait()

wall = time.perf_counter() - t_start
assert rc == 0, f'ffmpeg thất bại (exit code {rc}) - xem log lỗi ở trên.'

n  = max(STATS['frames'], 1)
ns = max(STATS['swapped'], 1)

print()
print(f'Xong: {STATS["frames"]} frame, trong đó {STATS["swapped"]} frame có mặt để swap '
      f'({STATS["swapped"] / n:.0%}).')
if hair_engine is not None:
    print(f'      thay tóc thành công {STATS["haired"]}/{STATS["swapped"]} frame có mặt '
          f'({STATS["haired"] / ns:.0%}).')
    if hair_engine.n_faded:
        print(f'      {hair_engine.n_faded} frame bị bỏ tóc do quay đầu quá nhiều '
              f'({hair_engine.n_faded / ns:.0%}) - nới HAIR_MAX_TURN nếu thấy tóc chớp tắt.')
    if hair_engine.n_inpainted:
        print(f'      {hair_engine.n_inpainted} frame phải inpaint tóc cũ thừa '
              f'({hair_engine.n_inpainted / ns:.0%}).')
print(f'Output: {final_output}')

# ---- Thời gian từng bước: để biết cái nào tốn, thay vì đoán ----
print()
print('Thời gian trung bình mỗi frame:')
print(f'  detect mặt : {STATS["detect"] / n * 1000:7.1f} ms')
print(f'  swap       : {STATS["swap"] / ns * 1000:7.1f} ms  (tính trên frame có mặt)')
print(f'  làm nét    : {STATS["enhance"] / ns * 1000:7.1f} ms'
      f'{"" if restorer is not None else "      (TẮT)"}')
print(f'  thay tóc   : {STATS["hair"] / ns * 1000:7.1f} ms'
      f'{"" if hair_engine is not None else "      (TẮT)"}')

busy = STATS['detect'] + STATS['swap'] + STATS['enhance'] + STATS['hair']
if restorer is not None:
    print(f'-> làm nét chiếm {STATS["enhance"] / max(busy, 1e-9):.0%} thời gian xử lý '
          f'(USE_GFPGAN = False để bỏ).')
if hair_engine is not None:
    print(f'-> thay tóc chiếm {STATS["hair"] / max(busy, 1e-9):.0%} thời gian xử lý '
          f'(HAIR_REMOVE_LEFTOVER = False để rẻ hơn).')

print()
print(f'Tổng: {wall:.1f}s cho {STATS["frames"]} frame ({wall / n * 1000:.0f} ms/frame, '
      f'{n / max(wall, 1e-9):.1f} fps xử lý).')

## 7. Kiểm tra kết quả

Audio đã được ghép ngay trong cell trên (ffmpeg nhận frame qua pipe và mux luôn audio gốc
trong cùng một pass), nên ở đây chỉ cần xác nhận file xuất ra hợp lệ.

In [ ]:
import os

assert os.path.exists(final_output) and os.path.getsize(final_output) > 0, \
    'Không tạo được video output — chạy lại cell xử lý video ở mục 6.'
print(f'{final_output}  —  {os.path.getsize(final_output) / 1e6:.1f} MB')
print()

# Xác nhận có stream video (và audio, nếu video gốc có audio)
!ffprobe -v error -show_entries stream=index,codec_type,codec_name,width,height,r_frame_rate,duration -of default=noprint_wrappers=1 {final_output}

## 8. Xem kết quả

In [ ]:
import os
from IPython.display import Video, display

size_mb = os.path.getsize(final_output) / 1e6
if size_mb > 50:
    # embed=True nhét toàn bộ file dưới dạng base64 vào output của notebook -> file .ipynb phình to
    # và trình duyệt dễ treo với video dài. Trường hợp đó thì tải về xem thay vì preview inline.
    print(f'Video {size_mb:.1f} MB — quá lớn để nhúng inline, chạy cell dưới để tải về máy.')
else:
    display(Video(final_output, embed=True, width=480))

In [ ]:
# Tải file về máy
from google.colab import files
files.download(final_output)

## Ghi chú / Giới hạn / Hướng cải thiện

### Cách chỉnh phần tóc cho nhanh

Đừng chạy cả video để thử. Vòng lặp đúng là: **sửa mục 0 → chạy lại cell mục 0 → chạy lại cell
6.1**. Ba giây một vòng. Chỉ chạy 6.2 khi đã hài lòng với frame preview (thử thêm vài giá trị
`PREVIEW_AT` khác nhau: 0.1 / 0.5 / 0.9 để bắt cả những đoạn quay đầu).

### Giới hạn đã biết của cách làm này (không phải bug)

- **Tóc-nguồn là ảnh 2D chính diện.** Phép đồng dạng chỉ xoay/phóng/tịnh tiến được nó. Đầu quay
  ra khỏi mặt phẳng ảnh thì không có cách nào dán đúng — nên có `HAIR_MAX_TURN` mờ tóc đi thay
  vì dán bừa. Muốn xử lý thật thì cần dựng tóc 3D (HairNet, NeuralHDHair) hoặc một mạng
  hair-transfer có điều kiện tư thế (**HairFastGAN**, **StyleYourHair**, **Barbershop**) — nặng
  hơn hẳn và không chạy nổi realtime trên T4 cho từng frame.
- **Không có vật lý.** Tóc thật bay theo chuyển động đầu, còn ở đây nó gắn cứng vào hộp sọ. Đầu
  lắc nhanh sẽ thấy "sai sai" dù không chỉ ra được sai ở đâu.
- **Tóc cũ nằm ngoài `HAIR_ROI` thì không bị xoá.** Vùng làm việc là một hộp quanh đầu; tóc dài
  quá hộp đó nằm ngoài tầm với của bước xoá tóc thừa. Thấy tóc cũ còn thò ra phía dưới/hai bên
  dù đã bật `HAIR_REMOVE_LEFTOVER` thì **tăng `HAIR_ROI`** (số thứ ba là bề xuống dưới) — đổi
  lại mỗi frame xử lý một vùng to hơn nên chậm hơn một chút.
- **`cv2.inpaint` từng frame là độc lập.** Vùng nền vá được có thể lăn tăn nhẹ giữa các frame.
  Nếu thấy rõ: tắt `HAIR_REMOVE_LEFTOVER` (đổi lại tóc cũ thò ra), hoặc chọn ảnh nguồn có tóc
  **dài hơn/dày hơn** tóc trong video — khi tóc mới trùm hết tóc cũ thì bước này không chạy.
- **Ảnh nguồn crop sát đầu** → mảng tóc có mép cắt thẳng. Cell mục 5b cảnh báo, nhưng không
  chữa được bằng tham số: phải đổi ảnh.
- **MediaPipe nhầm lớp** khi tóc và nền cùng màu (tóc đen trên nền tối), khi đội mũ, hoặc tóc
  buộc/che bởi tay. Xem trực tiếp mask ở cell preview mục 5b.

### Chọn ảnh nguồn thế nào cho ăn

Thấy **hết** mái tóc (đừng crop sát mặt) · chính diện · nền tương phản với màu tóc · ánh sáng
đều · độ phân giải cao hơn cái đầu trong video (warp thu nhỏ thì nét, phóng to thì nhoè) ·
không đội mũ, không buộc tóc bằng phụ kiện to.

### Phần còn lại (giữ từ bản gốc)

- **Bật/tắt làm nét / thay tóc**: đổi ở **mục 0** rồi chạy lại từ đầu. Đổi giữa chừng từ `False`
  sang `True` thì phải chạy lại mục 1 (cài package), mục 2 (tải weights) và mục 5/5b — vì lần
  chạy trước đã bỏ qua chúng.
- **Video nhiều người mà muốn thay người khác, không phải người to nhất**: sửa `pick_face()` ở
  mục 6.0. Cần thay **hai người khác nhau** thì đây là notebook sai — dùng
  `video_face_swap_kiss.ipynb`, nó có sẵn `FaceTracker` khớp danh tính ArcFace.
- **Flicker giữa các frame**: face swap từng frame độc lập nên đôi khi có giật/nhoè theo thời
  gian. GFPGAN tự nó cũng gây flicker vì "sáng tác" chi tiết khác nhau mỗi frame — tắt đi đôi
  khi lại đỡ giật hơn. Phần tóc đã có EMA (`HAIR_SMOOTH`, `HAIR_MASK_SMOOTH`), nhưng mặt thì
  chưa; muốn làm thì smoothing landmark giữa các frame liền kề.
- **`inswapper_128`**: link tải có thể thay đổi do vấn đề chính sách/gỡ bỏ. Cell mục 2 tự kiểm
  tra dung lượng file và báo lỗi ngay nếu tải hỏng.